In [ ]:
# Pair Construction (MNIST)
Create a balanced training subset (100 images per digit) and generate:
- positive pairs: same class
- negative pairs: different class

In [ ]:
# Load + subset
import numpy as np
from tensorflow.keras.datasets import mnist

(xtr, ytr), (xte, yte) = mnist.load_data()

# pick 100 per class
idx = []
for c in range(10):
    c_idx = np.where(ytr == c)[0]
    idx.extend(np.random.choice(c_idx, size=100, replace=False))
idx = np.array(idx)

x_small = xtr[idx]
y_small = ytr[idx]

# normalize + flatten
x_small = (x_small / 255.0).reshape(-1, 784)

print(x_small.shape, y_small.shape)

In [ ]:
# Pair generator
def make_pairs(x, y, neg_ratio=1.0, seed=42):
    rng = np.random.default_rng(seed)
    pairs = []
    labels = []

    # group indices by class
    by_class = {c: np.where(y == c)[0] for c in range(10)}

    # positive pairs
    for c in range(10):
        ids = by_class[c]
        for i in range(len(ids)-1):
            a = ids[i]
            b = ids[i+1]
            pairs.append([x[a], x[b]])
            labels.append(1)

    # negative pairs: sample across different classes
    n_pos = sum(labels)
    n_neg = int(neg_ratio * n_pos)

    classes = np.arange(10)
    for _ in range(n_neg):
        c1, c2 = rng.choice(classes, size=2, replace=False)
        a = rng.choice(by_class[c1])
        b = rng.choice(by_class[c2])
        pairs.append([x[a], x[b]])
        labels.append(0)

    pairs = np.array(pairs)          # (N, 2, 784)
    labels = np.array(labels).astype(int)
    return pairs, labels

pairs, lbls = make_pairs(x_small, y_small, neg_ratio=1.0)
print(pairs.shape, lbls.mean())